#  Week 2, Day 4 — May 28, 2026
## IQR Outlier Treatment — Clip Extreme Values

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {
    "raw":            BASE_DIR + "/data/raw",
    "processed":      BASE_DIR + "/data/processed",
    "metadata":       BASE_DIR + "/data/metadata",
    "notebooks":      BASE_DIR + "/notebooks",
    "visualizations": BASE_DIR + "/visualizations"
}
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 5, 30, 65, 95
YEAR_MIN, YEAR_MAX = 2015, 2026

In [ ]:
df_plastic = pd.read_csv(DIRS['processed'] + '/plastic_clean_v2.csv', parse_dates=['Date'])
df_species = pd.read_csv(DIRS['processed'] + '/species_clean_v2.csv')

# Load pre-computed IQR fences from Week 1
with open(DIRS['metadata'] + '/iqr_fences.json') as f:
    fences = json.load(f)

print('Loaded IQR fences:')
for dataset, cols in fences.items():
    print(f'  {dataset}:')
    for col, vals in cols.items():
        print(f'    {col}: [{vals["lower_fence"]}, {vals["upper_fence"]}]  ({vals["n_outliers"]} outliers)')

## Step 1: Plastic — No Clipping Needed

In [ ]:
# Plastic_Weight_kg had 0 outliers in May 23 analysis
# Plastic_Density_kg_km2 is derived — check it
from scipy import stats as scipy_stats

density = df_plastic['Plastic_Density_kg_km2']
q1, q3 = density.quantile(0.25), density.quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
outliers = df_plastic[(df_plastic['Plastic_Density_kg_km2'] < lower) | 
                       (df_plastic['Plastic_Density_kg_km2'] > upper)]

print(f'Plastic_Density_kg_km2 IQR analysis:')
print(f'  Fences: [{lower:.6f}, {upper:.6f}]')
print(f'  Outliers: {len(outliers)} / {len(df_plastic)}')
print()

if len(outliers) == 0:
    print('  No outliers in derived density column — no clipping needed')
    print('  Original values preserved (high-density hotspot readings are valid)')
    df_plastic_treated = df_plastic.copy()
else:
    print(f'  Clipping {len(outliers)} outlier records to fence boundaries')
    df_plastic_treated = df_plastic.copy()
    df_plastic_treated['Plastic_Density_kg_km2'] = df_plastic_treated['Plastic_Density_kg_km2'].clip(lower, upper)

## Step 2: Species — Filter Year Anomalies (Better Than IQR Clip)

In [ ]:
before = len(df_species)
df_species_treated = df_species[df_species['year'].between(YEAR_MIN, YEAR_MAX)].copy()
after = len(df_species_treated)

print(f'Species year filtering (2015–2026):')
print(f'  Before: {before:,} records')
print(f'  After : {after:,} records')
print(f'  Removed: {before-after:,} records with anomalous years')
print()
print('Year distribution after filter:')
print(df_species_treated['year'].value_counts().sort_index())

In [ ]:
# Save treated datasets
df_plastic_treated.to_csv(DIRS['processed'] + '/plastic_clean_final.csv', index=False)
df_species_treated.to_csv(DIRS['processed'] + '/species_clean_final.csv', index=False)
print('Final treated datasets saved ')
print(f'  plastic_clean_final.csv: {df_plastic_treated.shape}')
print(f'  species_clean_final.csv: {df_species_treated.shape}')